In [1]:
from elmo_data import Batcher
import torch
from torch import nn
import os, random
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.tokenize import TreebankWordTokenizer
from datasets import load_dataset

def reset_random_seeds():
    os.environ['PYTHONHASHSEED']=str(2)
    torch.manual_seed(2)
    np.random.seed(2)
    random.seed(2)
    torch.cuda.manual_seed(2)
    torch.cuda.manual_seed_all(2)

In [2]:
import re
from string import punctuation
from tqdm.auto import tqdm
import contractions as con

def preprocess_text(x):

    x = str(x).replace('&amp;','and').replace('&quot;','').lower()
    x = re.sub(r'@[A-Za-z0-9_]{1,15}', 'USER', x)
    x = con.fix(x)
    x = re.sub(r'&#x[0-9A-Fa-f]+;','',x)
    x = re.sub(r'&#\d+;',"'",x)
    x = re.sub(r'[^\x00-\x7F]+', "'",x)
    
    url_pattern = r'http\S+|www\S+'
    x = re.sub(url_pattern, 'LINK', x)
    
    punct_to_keep = """!,.:#?"-;//%$(){}@^*+<=>\\|'"""
    punct = ''.join([p for p in punctuation if p not in punct_to_keep])
    trans = str.maketrans(punct, ' ' * len(punct))
    x = x.translate(trans)
    x = ''.join(x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])\s*\1+', r'\1', x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])', r' \1 ', x)
    x = re.sub(r'\s+', ' ', x).strip().replace("'s "," 's ")
    x = x.replace("\\'"," '").replace("'"," ' ")
    
    return re.sub(r'\s+', ' ', x).strip()

In [3]:
class SelfAttention(nn.Module):

    def __init__(self,d_model,n_heads,look_ahead_mask=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.attn_scores = None
        self.d_head = self.d_model // self.n_heads
        self.look_ahead_mask = look_ahead_mask

        self.q = nn.Linear(self.d_model,self.d_model,False)
        self.k = nn.Linear(self.d_model, self.d_model, False)
        self.v = nn.Linear(self.d_model, self.d_model, False)

        self.attn_drop = nn.Dropout(0.1)
        self.lin = nn.Linear(d_model,d_model)
        self.lin_drop = nn.Dropout(0.1)

    def _split_heads(self,x):
        x = torch.reshape(x,shape=(x.shape[0],x.shape[1],self.n_heads,self.d_head))
        return torch.permute(x,(0,2,1,3))

    def forward(self,x):

        x,mask = x
        q,k,v = x
        mask = mask[:,torch.newaxis,torch.newaxis,:]
        maxlen = q.shape[1]
        b = q.shape[0]

        q = self.q(q)
        k = self.k(k)
        v = self.v(v)

        q = self._split_heads(q)
        k = self._split_heads(k)
        v = self._split_heads(v)

        k_trans = torch.permute(k,(0,1,3,2))
        attn = torch.matmul(q,k_trans) / (self.d_head ** 0.5)
        attn = torch.where(mask,-1e8,attn)

        if self.look_ahead_mask:
            look_ahead_mask = torch.triu(torch.ones(maxlen, maxlen, device=attn.device),
                                         diagonal=1).bool()
            attn = attn.masked_fill(look_ahead_mask, -1e8)

        attn = torch.softmax(attn,-1)
        self.attn_scores = attn
        attn = self.attn_drop(attn)

        x = torch.matmul(attn,v)
        x = x.permute(0, 2, 1, 3)
        x = torch.reshape(x,(b,maxlen,self.d_model))
        return x


In [4]:
from collections import Counter

class WordTokenizer:

    def __init__(self, vocab_size=10000):

        self.vocab_size = vocab_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"

        self.special_tokens = [
            self.pad_token,
            self.unk_token,
            self.sos_token,
            self.eos_token
        ]

        self.word2id = {}
        self.id2word = {}

    def fit(self, texts):

        counter = Counter()

        for text in texts:
            tokens = text.split()
            counter.update(tokens)

        most_common = counter.most_common(
            self.vocab_size - len(self.special_tokens)
        )

        vocab_words = [w for w, _ in most_common]

        vocab = self.special_tokens + vocab_words

        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, text, add_special_tokens=True):

        tokens = text.split()

        ids = []

        if add_special_tokens:
            ids.append(self.word2id[self.sos_token])

        for token in tokens:
            ids.append(
                self.word2id.get(token, self.word2id[self.unk_token])
            )

        if add_special_tokens:
            ids.append(self.word2id[self.eos_token])

        return ids

    def decode(self, ids, remove_special=True):

        words = []

        for i in ids:
            w = self.id2word.get(i, self.unk_token)

            if remove_special and w in self.special_tokens:
                continue

            words.append(w)

        return " ".join(words)

    def pad(self, sequences, max_len=None):

        if max_len is None:
            max_len = max(len(s) for s in sequences)

        padded = []

        for seq in sequences:

            seq = seq[:max_len]

            padding = [self.word2id[self.pad_token]] * (max_len - len(seq))

            padded.append(seq + padding)

        return padded

    def encode_batch(self, texts, pad=True, max_len=None):

        sequences = [self.encode(t) for t in texts]

        if pad:
            sequences = self.pad(sequences, max_len)

        return sequences

In [5]:
class AttentionPool(nn.Module):
    
    def __init__(self,dim):
        super().__init__()
        self.dim = dim 
        self.score = None
        self.w = nn.Linear(dim,1,bias=False)
        
    def forward(self,x,mask=None):
        attn = self.w(x)
        if mask is not None:
            mask = mask[:,:,torch.newaxis]
            attn = torch.where(mask,-1e8,attn)
        attn = torch.softmax(attn,axis=1)
        self.score = attn
        x = x * attn
        return x.sum(1)

In [6]:
from collections import Counter

class WordTokenizer:

    def __init__(self, vocab_size=10000):

        self.vocab_size = vocab_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"

        self.special_tokens = [
            self.pad_token,
            self.unk_token,
            self.sos_token,
            self.eos_token
        ]

        self.word2id = {}
        self.id2word = {}

    def fit(self, texts):

        counter = Counter()

        for text in texts:
            tokens = text.split()
            counter.update(tokens)

        most_common = counter.most_common(
            self.vocab_size - len(self.special_tokens)
        )

        vocab_words = [w for w, _ in most_common]

        vocab = self.special_tokens + vocab_words

        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, text, add_special_tokens=True):

        tokens = text.split()

        ids = []

        if add_special_tokens:
            ids.append(self.word2id[self.sos_token])

        for token in tokens:
            ids.append(
                self.word2id.get(token, self.word2id[self.unk_token])
            )

        if add_special_tokens:
            ids.append(self.word2id[self.eos_token])

        return ids

    def decode(self, ids, remove_special=True):

        words = []

        for i in ids:
            w = self.id2word.get(i, self.unk_token)

            if remove_special and w in self.special_tokens:
                continue

            words.append(w)

        return " ".join(words)

    def pad(self, sequences, max_len=None):

        if max_len is None:
            max_len = max(len(s) for s in sequences)

        padded = []

        for seq in sequences:

            seq = seq[:max_len]

            padding = [self.word2id[self.pad_token]] * (max_len - len(seq))

            padded.append(seq + padding)

        return padded

    def encode_batch(self, texts, pad=True, max_len=None):

        sequences = [self.encode(t) for t in texts]

        if pad:
            sequences = self.pad(sequences, max_len)

        return sequences

In [7]:
def load_glove_model(path, dim=50):
    glove_model = {}

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split()
            
            if len(parts) < dim + 1:
                continue

            word = parts[0]
            vector = parts[-dim:] 

            try:
                embedding = np.array(vector, dtype=np.float32)
                glove_model[word] = embedding
            except ValueError:
                continue

    print(f"{len(glove_model)} words loaded!")
    return glove_model

In [1]:
file = "/media/bibek/New Volume/dolma_300_2024_1.2M.100_combined.txt"

emb = load_glove_model(file,dim=300)

In [14]:
def initialize_embeddings(vocab_size,embed_dim = 300):

    limit = np.sqrt(6 / (embed_dim + embed_dim))
    w_emb = np.random.uniform(-limit, limit, (vocab_size, embed_dim))

    for w,idx in tok.word2id.items():
        try: w_emb[idx] = emb[w]
        except: pass
        
    w_emb = torch.tensor(w_emb,dtype=torch.float32)
        
    return w_emb
    

In [15]:
from torchinfo import summary
from elmo_layers import ELMoLayer


class ELMoClassifier(nn.Module):
    
    def __init__(self,vocab,glove_emb,dim=128,elmo_dim=256,emb_dim=300):
        
        super().__init__()
        self.dim = dim
        self.vocab = vocab
        self.emb = nn.Embedding(vocab,emb_dim,_weight=glove_emb)
        self.emb.requires_grad_(requires_grad=False)
        self.rnn = nn.GRU(emb_dim + elmo_dim,dim,
                          batch_first=True)
        self.out = nn.Linear(dim,n_classes) 
        self.elmo = ELMoLayer()
        self.elmo.load_state_dict(torch.load("elmo_layer.pth"))
        self.pool = AttentionPool(dim)
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.attn = SelfAttention(128,1)
        
    def forward(self,x,x_,test=False):
        mask = x == 0
        elmo_x = self.elmo(x_)
        x = self.emb(x)
        x = torch.concat([x,elmo_x],dim=-1)
        rnn,_ = self.rnn(x) 
        
        x = self.pool(rnn,mask=mask)
        
        if test:
            return rnn
        
        return self.out(x)


In [16]:

import numpy as np

tok = WordTokenizer(10000)

def prepare_data(train_data,test_data,val_data,tok,elmo=False):

    xtrain,ytrain = train_data
    xtest,ytest = test_data
    xval,yval = val_data
    
    if elmo:

        elmo_train = [tree_bank.tokenize(x) for x in xtrain]
        elmo_test = [tree_bank.tokenize(x) for x in xtest]
        elmo_val = [tree_bank.tokenize(x) for x in xval]

        elmo_vocab_file = "vocab-2016-09-10.txt"
        batcher = Batcher(elmo_vocab_file,20,)
        elmo_train = batcher.batch_sentences(elmo_train)
        elmo_test = batcher.batch_sentences(elmo_test)
        elmo_val = batcher.batch_sentences(elmo_val)

    tok.fit(xtrain)
    xtrain = tok.encode_batch(xtrain)
    xtest = tok.encode_batch(xtest)
    xval = tok.encode_batch(xval)

    xtrain = np.array(xtrain)
    xtest = np.array(xtest)
    xval = np.array(xval)

    ytest = ytest.astype(np.int32)
    ytrain = ytrain.astype(np.int32)
    yval = yval.astype(np.int32)

    vocab = len(tok.word2id)
    
    data = {}
    
    if elmo:
        
        data['train'] = (xtrain,elmo_train,ytrain)
        data['test'] = (xtest,elmo_test,ytest)
        data['val'] = (xval,elmo_val,yval)
    else:
        data['train'] = (xtrain,ytrain)
        data['test'] = (xtest,ytest)
        data['val'] = (xval,yval)
    
    return vocab,data


In [17]:
from tqdm.auto import tqdm

def train_model(data,model,epochs=7):
    
    train,valid = data

    losses = {'train':[],'valid':[]}

    for e in range(1,epochs + 1):

        print(f"{e}/{epochs}")

        loss = 0
        model.train()

        for i,batch in enumerate(tqdm(train),1):

            lr = lr_scheduler[i]
            opt.param_groups[0]['lr'] = lr

            opt.zero_grad()

            x,z,y = batch
            if n_classes == 1:
                y = y.float().unsqueeze(1)
            elif n_classes > 1:
                y = y.long()
            pred = model(x,z)
            batch_loss = loss_fn(pred,y)
            batch_loss.backward()
            opt.step()

            loss += batch_loss.item()

        loss = round(loss / len(train),4)
        losses['train'].append(loss)

        print(f'Train Loss = {loss}')

        model.eval()
        loss = 0

        with torch.no_grad():

            for i,batch in enumerate(valid):

                x,z,y = batch
                if n_classes == 1:
                    y = y.float().unsqueeze(1)
                elif n_classes > 1:
                    y = y.long()
                pred = model(x,z)
                batch_loss = loss_fn(pred,y)
                loss += batch_loss.item()

        loss = round(loss / len(test),4)
        print(f"Val Loss = {loss}")


        if e == 1 or loss < min(losses['valid']):
            torch.save(model.state_dict(),'saved_weights.pt')
            print("--weights saved")

        losses['valid'].append(loss)

        print()

In [18]:
from sklearn.metrics import classification_report,f1_score,confusion_matrix,recall_score,accuracy_score
import seaborn as sb


def run_metrics(model,data,n_classes):
    
    test,ytest = data

    if n_classes == 1:

        preds = []

        for x,z,_ in test:
            pred = model(x,z)
            pred = torch.sigmoid(pred)
            pred = torch.round(pred).reshape(-1).detach().cpu().numpy()
            preds.extend(pred)

        print("f1 score :",round(f1_score(ytest,preds),4))
        print("recall score :",round(recall_score(ytest,preds),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

    elif n_classes > 1:

        preds = []

        for x,z,_ in test:
            pred = model(x,z)
            pred = torch.softmax(pred,-1)
            pred = torch.argmax(pred,1).cpu().numpy()
            preds.extend(pred)
        print("f1 score :",round(f1_score(ytest,preds,average='macro'),4))
        print("recall score :",round(recall_score(ytest,preds,average='macro'),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

In [19]:
tree_bank = TreebankWordTokenizer()

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("holdback.csv")
df = pd.concat([df_train,df_test]).reset_index(drop=True)

X = df.samples.apply(lambda x: preprocess_text(x))
Y = df.targets.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [20]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [21]:
glove_emb = initialize_embeddings(vocab)

In [22]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)


In [23]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 4

def create_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [24]:
train_model((train,valid),model)

1/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 1.0016
Val Loss = 0.5305
--weights saved

2/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.9291
Val Loss = 0.5194
--weights saved

3/7


...

Train Loss = 0.7838
Val Loss = 0.4772
--weights saved

6/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.7385
Val Loss = 0.5545

7/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.6887
Val Loss = 0.5482



In [25]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [29]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.6145
recall score : 0.6075
accuracy score : 0.606


In [27]:
tree_bank = TreebankWordTokenizer()

df = pd.read_csv("/mnt/38DEEB97DEEB4C26/dataset/train.csv")

X = df.text.apply(lambda x: preprocess_text(x)).values
Y = df.target.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [28]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [29]:
glove_emb = initialize_embeddings(vocab)

In [30]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [31]:
BATCH = 8

def create_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [32]:
train_model((train,valid),model)

1/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.4899
Val Loss = 0.2339
--weights saved

2/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.4223
Val Loss = 0.227
--weights saved

3/7


...

Train Loss = 0.3679
Val Loss = 0.2263
--weights saved

6/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.3457
Val Loss = 0.2428

7/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.3263
Val Loss = 0.2421



In [30]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.766
recall score : 0.7312
accuracy score : 0.815


In [34]:
from datasets import load_dataset

tree_bank = TreebankWordTokenizer()

data = load_dataset("tweet_eval", "sentiment")
xtrain,ytrain = pd.DataFrame(data['train']).values.T
xtrain = [preprocess_text(x) for x in xtrain]


xtest,ytest = pd.DataFrame(data['test']).values.T
xtest = [preprocess_text(x) for x in xtest]

xval,yval = pd.DataFrame(data["validation"]).values.T
xval = [preprocess_text(x) for x in xval]

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [35]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [36]:
glove_emb = initialize_embeddings(vocab)

In [37]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [38]:
BATCH = 32

def create_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [39]:
train_model((train,valid),model)

1/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.793
Val Loss = 0.1279
--weights saved

2/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.7313
Val Loss = 0.1236
--weights saved

3/7


...

Train Loss = 0.6581
Val Loss = 0.1172
--weights saved

6/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.6396
Val Loss = 0.1176

7/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.6186
Val Loss = 0.1165
--weights saved



In [40]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [31]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.631
recall score : 0.636
accuracy score : 0.638


In [42]:
tree_bank = TreebankWordTokenizer()

df_train = pd.read_csv("trec50_train.csv")
df_test = pd.read_csv("trec50_test.csv")

X = df_train.text.apply(lambda x: preprocess_text(x)).values
Y = df_train['label-fine'].values

xtrain,xval,ytrain,yval = train_test_split(X,Y,train_size=0.9,random_state=0)
xtest = df_test.text.apply(lambda x: preprocess_text(x)).values
ytest = df_test['label-fine'].values


In [43]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [44]:
glove_emb = initialize_embeddings(vocab)

In [45]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [46]:
BATCH = 8

def create_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [47]:
train_model((train,valid),model)

1/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 2.4606
Val Loss = 2.2143
--weights saved

2/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 1.6293
Val Loss = 1.6999
--weights saved

3/7


...

Train Loss = 0.7531
Val Loss = 1.1313
--weights saved

6/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 0.5965
Val Loss = 1.1192
--weights saved

7/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 0.4848
Val Loss = 1.0922
--weights saved



In [48]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [32]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.58
recall score : 0.636
accuracy score : 0.794


In [50]:
df = pd.read_csv("/mnt/38DEEB97DEEB4C26/dataset/IMDB Dataset.csv")

X = df.review.apply(lambda x: preprocess_text(x)).values
df.sentiment = df.sentiment.map({'positive':1,'negative':0})
Y = df.sentiment.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [51]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [52]:
glove_emb = initialize_embeddings(vocab)

In [53]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [54]:
BATCH = 32

def create_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [55]:
train_model((train,valid),model)

1/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.4607
Val Loss = 0.212
--weights saved

2/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3924
Val Loss = 0.2044
--weights saved

3/7


...

Train Loss = 0.3349
Val Loss = 0.1881
--weights saved

6/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3189
Val Loss = 0.1864
--weights saved

7/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.304
Val Loss = 0.1882



In [56]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [34]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.8497
recall score : 0.8592
accuracy score : 0.8501
